In [1]:
import json
import os
import sys

CURRENT_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, os.pardir))
LLM_DIR = os.path.join(PROJECT_ROOT, "llm_results")  # ou "llm_results/arara" se for o seu caso
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [2]:
def is_candidate(file_name: str, eval_type: str) -> bool:
    return (
        eval_type in file_name
        and ("output" in file_name or "prediction" in file_name)
        and file_name.endswith(".jsonl")
    )

def validate_predictions_file(path: str):
    if not os.path.isfile(path):
        return False, "file does not exist"
    if os.path.getsize(path) == 0:
        return False, "empty file (0 bytes)"
    # read the first non-empty line
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception as e:
                return False, f"invalid JSON on the 1st non-empty line: {e}"
            if not isinstance(obj, dict):
                return False, "1st line is not a JSON object"
            if "id" not in obj or "response" not in obj:
                return False, "missing required keys: 'id' and/or 'response'"
            return True, "ok"
    return False, "only empty lines"

In [3]:

database = 'neo4j'
query_type = 'condition'
dataset_type = 'MisinformedQuery'  # ou 'ExplicitQuery' dependendo do seu caso
groundtruths = os.path.join(PROJECT_ROOT, "dataset", "book", f"{dataset_type}.json")
eval_type = f"book-{dataset_type}"

comparisons = {
    f"arara_openrouter_claude35/book-MisinformedQuery_arara_openrouter_claude35_historyTrue-prediction.jsonl": 
    f"claude-3-5-sonnet-20241022/book-MisinformedQuery_claude-3-5-sonnet-20241022-historyFalse-prediction.jsonl",
    f"arara_openrouter_claude35/book-MisinformedQuery_arara_openrouter_claude35_None-prediction.jsonl": 
    f"claude-3-5-sonnet-20241022/book-MisinformedQuery_claude-3-5-sonnet-20241022-historyTrue-prediction.jsonl",
}

def run_eval(predictions, ids):
    scrit_name = (
            f'eval_book.py --database {database} '
            f'--query_type {query_type} '
            f'--groundtruths "{groundtruths}" '
            f'--predictions "{predictions}" '
            f"--ids '{ids}'"
        )
    get_ipython().run_line_magic('run', scrit_name)
    print('---------------------------------------------')

for folderA, folderB in comparisons.items():
    arara_file = os.path.join(LLM_DIR, folderA)
    is_valid, message = validate_predictions_file(arara_file)
    if not is_valid:
        print(f"Skipping {folderA}: {message}")
        continue

    baseline_file = os.path.join(LLM_DIR, folderB)
    is_valid, message = validate_predictions_file(baseline_file)
    if not is_valid:
        print(f"Skipping {folderB}: {message}")
        continue

    predictions = []
    with open(arara_file, "r") as file:
        for line in file:
            predictions.append(json.loads(line))
    ids = [predictions[i]['id'] for i in range(len(predictions))]

    print("Arara file:", arara_file)
    run_eval(arara_file, ids)
    
    print("Baseline file:", baseline_file)
    run_eval(baseline_file, ids)
    print("="*100)
    


Arara file: /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results/arara_openrouter_claude35/book-MisinformedQuery_arara_openrouter_claude35_historyTrue-prediction.jsonl


INFO:root:Successfully connected to the Neo4j database.
Evaluating predictions: 100%|██████████| 200/200 [00:03<00:00, 56.66it/s]
INFO:root:Successfully connected to the Neo4j database.


 & .12 & .42 & .33 & .43 & .12
 Ftr: 0.115
 Recall: 0.42246428571428574
 Precision: 0.33009126984126985
 Ndcg: 0.42738608938769196
 Satisfied Ratio: 0.11538461538461539
---------------------------------------------
Baseline file: /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results/claude-3-5-sonnet-20241022/book-MisinformedQuery_claude-3-5-sonnet-20241022-historyFalse-prediction.jsonl


Evaluating predictions: 100%|██████████| 200/200 [00:09<00:00, 20.15it/s]
INFO:root:Successfully connected to the Neo4j database.


 & .04 & .09 & .03 & .06 & .04
 Ftr: 0.045
 Recall: 0.09261904761904761
 Precision: 0.03179779942279942
 Ndcg: 0.06017487474524742
 Satisfied Ratio: 0.04
---------------------------------------------
Arara file: /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results/arara_openrouter_claude35/book-MisinformedQuery_arara_openrouter_claude35_None-prediction.jsonl


Evaluating predictions: 100%|██████████| 200/200 [00:03<00:00, 64.46it/s]
INFO:root:Successfully connected to the Neo4j database.


 & .22 & .49 & .42 & .48 & .15
 Ftr: 0.22
 Recall: 0.49304166666666666
 Precision: 0.4224047619047619
 Ndcg: 0.4815434493951215
 Satisfied Ratio: 0.15
---------------------------------------------
Baseline file: /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results/claude-3-5-sonnet-20241022/book-MisinformedQuery_claude-3-5-sonnet-20241022-historyTrue-prediction.jsonl


Evaluating predictions: 100%|██████████| 200/200 [00:07<00:00, 25.73it/s]

 & .01 & .05 & .02 & .04 & .09
 Ftr: 0.01
 Recall: 0.04808333333333334
 Precision: 0.019680555555555555
 Ndcg: 0.04155593192212888
 Satisfied Ratio: 0.09024943310657596
---------------------------------------------


In [4]:
"""
database = 'neo4j'
query_type = 'condition'
dataset_type = 'ImplicitQuery'  # ou 'ExplicitQuery' dependendo do seu caso
groundtruths = os.path.join(PROJECT_ROOT, "dataset", "movie", f"{dataset_type}.json")
eval_type = f"movie-{dataset_type}"

comparisons = {
    f"arara_openrouter_claude35/movie-ImplicitQuery_arara_openrouter_claude35_historyTrue-prediction.jsonl": f"claude-3-5-sonnet-20241022/movie-ImplicitQuery_claude-3-5-sonnet-20241022-prediction.jsonl",
    f"arara_openrouter_claude35/movie-ImplicitQuery_arara_openrouter_claude35_None-prediction.jsonl": f"claude-3-5-sonnet-20241022/movie-ImplicitQuery_claude-3-5-sonnet-20241022-historyTrue-prediction.jsonl",
}

def run_eval(predictions, ids):
    scrit_name = (
            f'eval_movie.py --database {database} '
            f'--query_type {query_type} '
            f'--groundtruths "{groundtruths}" '
            f'--predictions "{predictions}" '
            f"--ids '{ids}'"
        )
    get_ipython().run_line_magic('run', scrit_name)
    print('---------------------------------------------')
    

for folderA, folderB in comparisons.items():
    arara_file = os.path.join(LLM_DIR, folderA)
    is_valid, message = validate_predictions_file(arara_file)
    if not is_valid:
        print(f"Skipping {folderA}: {message}")
        continue

    baseline_file = os.path.join(LLM_DIR, folderB)
    is_valid, message = validate_predictions_file(baseline_file)
    if not is_valid:
        print(f"Skipping {folderB}: {message}")
        continue

    predictions = []
    with open(arara_file, "r") as file:
        for line in file:
            predictions.append(json.loads(line))
    ids = [predictions[i]['id'] for i in range(len(predictions))]

    print("Arara file:", arara_file)
    run_eval(arara_file, ids)
    
    #print("Baseline file:", baseline_file)
    #run_eval(baseline_file, ids)
    #print("="*100)
    break
"""

'\ndatabase = \'neo4j\'\nquery_type = \'condition\'\ndataset_type = \'ImplicitQuery\'  # ou \'ExplicitQuery\' dependendo do seu caso\ngroundtruths = os.path.join(PROJECT_ROOT, "dataset", "movie", f"{dataset_type}.json")\neval_type = f"movie-{dataset_type}"\n\ncomparisons = {\n    f"arara_openrouter_claude35/movie-ImplicitQuery_arara_openrouter_claude35_historyTrue-prediction.jsonl": f"claude-3-5-sonnet-20241022/movie-ImplicitQuery_claude-3-5-sonnet-20241022-prediction.jsonl",\n    f"arara_openrouter_claude35/movie-ImplicitQuery_arara_openrouter_claude35_None-prediction.jsonl": f"claude-3-5-sonnet-20241022/movie-ImplicitQuery_claude-3-5-sonnet-20241022-historyTrue-prediction.jsonl",\n}\n\ndef run_eval(predictions, ids):\n    scrit_name = (\n            f\'eval_movie.py --database {database} \'\n            f\'--query_type {query_type} \'\n            f\'--groundtruths "{groundtruths}" \'\n            f\'--predictions "{predictions}" \'\n            f"--ids \'{ids}\'"\n        )\n    get

In [5]:
'''
database = 'neo4j'
query_type = 'condition'
dataset_type = 'ImplicitQuery'  # ou 'ExplicitQuery' dependendo do seu caso
groundtruths = os.path.join(PROJECT_ROOT, "dataset", "movie", f"{dataset_type}.json")

# --- use ---
# garanta que dataset_type exista (ex.: "ImplicitQuery" ou "ExplicitQuery")
# dataset_type = "ImplicitQuery"
eval_type = f"movie-{dataset_type}"

avaliados = 0
pulados = 0

for root, dirs, files in os.walk(LLM_DIR):
    print("root=", root)
    for file in files:
        print("file=", file)
        if not is_candidate(file, eval_type):
            continue

        predictions = os.path.join(root, file)
        ok, msg = validate_predictions_file(predictions)
        if not ok:
            print(f"🟡 Pulando: {predictions} — {msg}")
            pulados += 1
            continue

        print(f"✅ Avaliando: {predictions}")
        scrit_name = (
            f'eval_movie.py --database {database} '
            f'--query_type {query_type} '
            f'--groundtruths "{groundtruths}" '
            f'--predictions "{predictions}"'
        )
        get_ipython().run_line_magic('run', scrit_name)
        print('---------------------------------------------')
        avaliados += 1
print(f"\nResumo: {avaliados} avaliados, {pulados} pulados.")
'''

'\ndatabase = \'neo4j\'\nquery_type = \'condition\'\ndataset_type = \'ImplicitQuery\'  # ou \'ExplicitQuery\' dependendo do seu caso\ngroundtruths = os.path.join(PROJECT_ROOT, "dataset", "movie", f"{dataset_type}.json")\n\n# --- use ---\n# garanta que dataset_type exista (ex.: "ImplicitQuery" ou "ExplicitQuery")\n# dataset_type = "ImplicitQuery"\neval_type = f"movie-{dataset_type}"\n\navaliados = 0\npulados = 0\n\nfor root, dirs, files in os.walk(LLM_DIR):\n    print("root=", root)\n    for file in files:\n        print("file=", file)\n        if not is_candidate(file, eval_type):\n            continue\n\n        predictions = os.path.join(root, file)\n        ok, msg = validate_predictions_file(predictions)\n        if not ok:\n            print(f"🟡 Pulando: {predictions} — {msg}")\n            pulados += 1\n            continue\n\n        print(f"✅ Avaliando: {predictions}")\n        scrit_name = (\n            f\'eval_movie.py --database {database} \'\n            f\'--query_type {q